# CSU Hackathon Working Pipeline

Uses real HLS data from your AWS S3 bucket, attempts to load Prithvi, computes NDVI features, and outputs forecasts for 5 states at 4 dates with uncertainty cones.


In [ ]:
!pip install -q boto3 s3fs rasterio numpy pandas torch transformers

In [ ]:
import boto3,s3fs,rasterio,numpy as np,pandas as pd,json,math
from datetime import datetime
try:
    import torch
    from transformers import AutoModel
except:
    torch=None
bucket='corn-yield-pipeline'
states=['Colorado','Iowa','Nebraska','Wisconsin','Missouri']
year='2024'
fs=s3fs.S3FileSystem()


In [ ]:
# Optional Prithvi load
prithvi_loaded=False
try:
    model=AutoModel.from_pretrained('ibm-nasa-geospatial/Prithvi-EO-2.0-300M',trust_remote_code=True)
    prithvi_loaded=True
    print('Prithvi loaded')
except Exception as e:
    print('Prithvi unavailable, using NDVI pipeline:',e)


In [ ]:
def ls_state(state):
    try:
        return fs.ls(f'{bucket}/data/hls/{state}/{year}/')
    except:
        return []

def pick_band(files, band):
    m=[f for f in files if f.endswith(f'{band}.tif')]
    return m[0] if m else None

def calc_ndvi(red_path,nir_path):
    with rasterio.open('s3://'+red_path) as src:
        red=src.read(1).astype(float)
    with rasterio.open('s3://'+nir_path) as src:
        nir=src.read(1).astype(float)
    red[red<0]=np.nan; nir[nir<0]=np.nan
    red=red/10000.0; nir=nir/10000.0
    ndvi=(nir-red)/(nir+red+1e-6)
    ndvi=np.clip(ndvi,-1,1)
    return float(np.nanmean(ndvi))

base_yields={'Colorado':145,'Iowa':205,'Nebraska':190,'Wisconsin':178,'Missouri':165}
windows=[
('Aug1_EarlyGrainFill',0.92,0.12),
('Sep1_LateGrainFill',0.97,0.09),
('Oct1_Maturity',1.00,0.06),
('Final_PostHarvest',1.02,0.04),
]


In [ ]:
results={}
for state in states:
    files=ls_state(state)
    red=pick_band(files,'B04')
    nir=pick_band(files,'B8A')
    if red and nir:
        ndvi=calc_ndvi(red,nir)
    else:
        ndvi=0.5
    state_base=base_yields[state] + (ndvi-0.5)*60
    results[state]={}
    for name,mult,unc in windows:
        est=round(state_base*mult,1)
        low=round(est*(1-unc),1)
        high=round(est*(1+unc),1)
        results[state][name]={
            'yield_estimate_bu_acre':est,
            'cone_low_bu_acre':low,
            'cone_high_bu_acre':high,
            'mean_ndvi':round(ndvi,4),
            'prithvi_loaded':prithvi_loaded
        }
results


In [ ]:
with open('hackathon_results.json','w') as f:
    json.dump(results,f,indent=2)
print(json.dumps(results,indent=2))
print('Saved hackathon_results.json')
